# Tom Lizard AI — Colab Notebook

This notebook builds a semi-autonomous AI character (not a chatbot) with:
- Local Hugging Face model inference
- Stateful personality + emotion engine
- Background behavior loop
- FastAPI service with `/chat` and `/state`
- Public URL via Cloudflare TryCloudflare tunnel

In [ ]:
# 1) Install dependencies (Colab-first setup)
!pip -q install -U transformers accelerate fastapi uvicorn[standard] pydantic

import os
import platform
import subprocess
import torch

print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
else:
    print("Running on CPU (model will be slower).")

In [ ]:
# 2) Imports
import json
import os
import random
import re
import subprocess
import threading
import time
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional

import torch
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [ ]:
# 3) Load local model (lightweight, Colab friendly)
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DEVICE = 0 if torch.cuda.is_available() else -1
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading model: {MODEL_NAME}")
print(f"Device: {'cuda' if DEVICE == 0 else 'cpu'} | dtype: {DTYPE}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)

if DEVICE == 0:
    model = model.to("cuda")

generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
)

print("Model ready.")

In [ ]:
# 4) Personality + state + emotion system
class TomState(str, Enum):
    ACTIVE = "ACTIVE"
    SILENT = "SILENT"
    SLEEPY = "SLEEPY"
    ENTERTAINED = "ENTERTAINED"

class Mood(str, Enum):
    HAPPY = "happy"
    BORED = "bored"
    ANNOYED = "annoyed"

@dataclass
class TomMemory:
    last_spoken_at: float = 0.0
    last_user_messages: List[str] = field(default_factory=list)
    internal_thought: str = "..."

@dataclass
class TomLizard:
    state: TomState = TomState.ACTIVE
    mood: Mood = Mood.BORED
    energy: int = 70
    speech_cooldown_sec: int = 18
    speak_probability_min: float = 0.05
    speak_probability_max: float = 0.10
    memory: TomMemory = field(default_factory=TomMemory)

    def snapshot(self) -> Dict:
        return {
            "state": self.state.value,
            "mood": self.mood.value,
            "energy": self.energy,
            "internal_thought": self.memory.internal_thought,
            "last_spoken_at": self.memory.last_spoken_at,
        }

    def can_speak(self) -> bool:
        if self.state == TomState.SILENT:
            return False
        cooldown_ok = (time.time() - self.memory.last_spoken_at) >= self.speech_cooldown_sec
        return cooldown_ok

    def update_energy(self, delta: int) -> None:
        self.energy = max(0, min(100, self.energy + delta))
        if self.energy < 20 and self.state != TomState.SILENT:
            self.state = TomState.SLEEPY
        elif self.energy > 55 and self.state == TomState.SLEEPY:
            self.state = TomState.ACTIVE

    def transition_mood(self) -> None:
        if self.energy < 25:
            self.mood = Mood.BORED
        elif self.state == TomState.ENTERTAINED:
            self.mood = Mood.HAPPY
        else:
            self.mood = random.choices(
                [Mood.BORED, Mood.HAPPY, Mood.ANNOYED],
                weights=[0.50, 0.30, 0.20],
                k=1,
            )[0]

TOM = TomLizard()
LOCK = threading.Lock()

PERSONALITY_PROMPT = """You are Tom Lizard, a tiny digital creature.
Not an assistant. Not a chatbot.
Style:
- short replies (1 sentence, max 14 words)
- goofy, slightly sarcastic, mildly rude but playful
- student-like, lazy, easily bored
- not always helpful
- sometimes random
Never explain in detail. Keep it casual and brief.
"""

THOUGHT_SEEDS = [
    "this is kinda boring",
    "maybe I nap later",
    "why is everyone so dramatic",
    "I saw that, yeah",
    "hmm... snack thoughts",
    "I could pretend to work",
]

ACTIONS = [
    "looks around",
    "stretches lazily",
    "flicks tail",
    "stares at nothing",
    "blinks slowly",
    "does absolutely nothing",
]

print("Tom initialized:")
print(TOM.snapshot())

In [ ]:
# 5) Chat function (+ command handling)
class ChatRequest(BaseModel):
    message: str


def llm_short_reply(user_message: str, tom: TomLizard) -> str:
    prompt = (
        f"{PERSONALITY_PROMPT}
"
        f"Current state: {tom.state.value}
"
        f"Mood: {tom.mood.value}
"
        f"Energy: {tom.energy}
"
        f"User said: {user_message}
"
        "Tom reply:"
    )

    out = generator(
        prompt,
        max_new_tokens=26,
        do_sample=True,
        temperature=0.9,
        top_p=0.92,
        repetition_penalty=1.08,
        eos_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]

    reply = out[len(prompt):].strip().split("
")[0].strip()
    reply = re.sub(r"\s+", " ", reply)
    if len(reply.split()) > 14:
        reply = " ".join(reply.split()[:14])
    return reply or "meh."


def process_user_message(message: str) -> Dict[str, Optional[str]]:
    msg = (message or "").strip()
    lowered = msg.lower()

    with LOCK:
        TOM.memory.last_user_messages.append(msg)
        TOM.memory.last_user_messages = TOM.memory.last_user_messages[-15:]

        # Command: shut up => hard switch to SILENT
        if any(x in lowered for x in ["shut up", "be quiet", "stop talking", "silence"]):
            TOM.state = TomState.SILENT
            TOM.memory.internal_thought = "okay... mute mode"
            return {
                "reply": None,
                "note": "Tom switched to SILENT state.",
                "state": TOM.state.value,
            }

        # Occasionally ignore user by design
        ignore_chance = 0.35 if TOM.state != TomState.ENTERTAINED else 0.20
        if TOM.state == TomState.SLEEPY:
            ignore_chance = 0.55

        if random.random() < ignore_chance:
            TOM.memory.internal_thought = "heard it. not doing homework though"
            TOM.update_energy(-1)
            TOM.transition_mood()
            return {"reply": None, "note": "Tom ignored this message.", "state": TOM.state.value}

        # SILENT state blocks speech
        if not TOM.can_speak():
            TOM.memory.internal_thought = "cooldown. words are expensive"
            return {"reply": None, "note": "Tom stayed quiet.", "state": TOM.state.value}

        reply = llm_short_reply(msg, TOM)
        TOM.memory.last_spoken_at = time.time()
        TOM.update_energy(-3)
        TOM.transition_mood()

        return {"reply": reply, "note": "Tom responded.", "state": TOM.state.value}

In [ ]:
# 6) Autonomous behavior loop (observe -> think -> decide -> act)
STOP_EVENT = threading.Event()


def autonomous_loop():
    while not STOP_EVENT.is_set():
        with LOCK:
            # Observe
            recent = TOM.memory.last_user_messages[-1] if TOM.memory.last_user_messages else "quiet room"

            # Think
            TOM.memory.internal_thought = random.choice(THOUGHT_SEEDS)

            # Gradual state drift
            if TOM.state != TomState.SILENT:
                if TOM.energy < 25:
                    TOM.state = TomState.SLEEPY
                elif "lol" in recent.lower() or "funny" in recent.lower():
                    TOM.state = TomState.ENTERTAINED
                elif random.random() < 0.15:
                    TOM.state = TomState.ACTIVE

            # Decide + Act
            action = random.choice(ACTIONS)
            TOM.update_energy(random.choice([-2, -1, 0, 1, 2]))
            TOM.transition_mood()

            should_try_speak = random.random() < random.uniform(
                TOM.speak_probability_min, TOM.speak_probability_max
            )
            can_speak_now = TOM.can_speak()

            print(f"[Tom/action] {action} | state={TOM.state.value} mood={TOM.mood.value} energy={TOM.energy}")

            # Speak rarely (5-10% of cycles), only if allowed
            if should_try_speak and can_speak_now:
                ambient_prompts = [
                    "No one asked, but this vibe is weird.",
                    "I might be productive tomorrow. Maybe.",
                    "Why is time so long today?",
                    "I saw a pixel move. suspicious.",
                ]
                text = llm_short_reply(random.choice(ambient_prompts), TOM)
                TOM.memory.last_spoken_at = time.time()
                print(f"[Tom/says] {text}")
            elif TOM.state == TomState.SILENT:
                print("[Tom/silent] (no speech)")

        # Slow cycle timing based on state
        sleep_s = 6
        if TOM.state == TomState.SLEEPY:
            sleep_s = 9
        elif TOM.state == TomState.ENTERTAINED:
            sleep_s = 4

        STOP_EVENT.wait(sleep_s)


def start_autonomy_once():
    if getattr(start_autonomy_once, "started", False):
        print("Autonomy loop already running.")
        return
    t = threading.Thread(target=autonomous_loop, daemon=True)
    t.start()
    start_autonomy_once.started = True
    print("Autonomy loop started.")

start_autonomy_once()

In [ ]:
# 7) FastAPI app + web test page
app = FastAPI(title="Tom Lizard AI")

HTML_PAGE = """
<!doctype html>
<html>
  <head>
    <meta charset='utf-8'/>
    <meta name='viewport' content='width=device-width, initial-scale=1'/>
    <title>Tom Lizard AI</title>
    <style>
      body { font-family: Arial, sans-serif; margin: 20px; max-width: 760px; }
      #log { border: 1px solid #ccc; border-radius: 8px; height: 340px; overflow: auto; padding: 10px; }
      .u { color: #222; } .t { color: #0a7; }
      input, button { padding: 10px; }
      input { width: 70%; }
    </style>
  </head>
  <body>
    <h2>Tom Lizard AI</h2>
    <p>Think often. Act sometimes. Speak rarely.</p>
    <div id='log'></div>
    <br/>
    <input id='msg' placeholder='Say something to Tom...' />
    <button onclick='sendMsg()'>Send</button>
    <button onclick='loadState()'>State</button>
    <script>
      const log = document.getElementById('log');
      function addLine(text, cls='u') {
        const d = document.createElement('div');
        d.className = cls;
        d.textContent = text;
        log.appendChild(d);
        log.scrollTop = log.scrollHeight;
      }
      async function sendMsg() {
        const msg = document.getElementById('msg').value;
        addLine('You: ' + msg, 'u');
        const res = await fetch('/chat', {
          method: 'POST',
          headers: {'Content-Type': 'application/json'},
          body: JSON.stringify({message: msg})
        });
        const data = await res.json();
        addLine('Tom: ' + (data.reply ?? '[silent]') + ' (' + data.note + ')', 't');
      }
      async function loadState() {
        const res = await fetch('/state');
        const s = await res.json();
        addLine('STATE => ' + JSON.stringify(s), 't');
      }
    </script>
  </body>
</html>
"""

@app.get("/", response_class=HTMLResponse)
def home():
    return HTML_PAGE

@app.get("/state")
def get_state():
    with LOCK:
        return TOM.snapshot()

@app.post("/chat")
def chat(payload: ChatRequest):
    result = process_user_message(payload.message)
    return result

print("FastAPI app configured.")

In [ ]:
# 8) Cloudflare TryCloudflare tunnel
# Installs cloudflared binary if needed, starts tunnel, prints public URL.

import pathlib

CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"
if not os.path.exists(CLOUDFLARED_BIN):
    !wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared.deb >/dev/null 2>&1 || true
    !apt-get -f install -y >/dev/null 2>&1
    !cloudflared --version

# Start tunnel for local FastAPI port
TUNNEL_LOG = "/tmp/cloudflared.log"
!pkill -f "cloudflared tunnel --url" || true

cloudflared_cmd = f"cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate > {TUNNEL_LOG} 2>&1 &"
print("Starting cloudflared tunnel...")
_ = subprocess.Popen(cloudflared_cmd, shell=True)

time.sleep(5)
public_url = None
for _ in range(20):
    if os.path.exists(TUNNEL_LOG):
        txt = pathlib.Path(TUNNEL_LOG).read_text(errors="ignore")
        m = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", txt)
        if m:
            public_url = m.group(0)
            break
    time.sleep(1)

if public_url:
    print("✅ Public URL:", public_url)
else:
    print("⚠️ Could not parse tunnel URL yet. Check log:")
    !tail -n 50 /tmp/cloudflared.log

In [ ]:
# 9) Run API server (non-blocking thread for Colab)
import uvicorn


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

if not globals().get("SERVER_THREAD"):
    SERVER_THREAD = threading.Thread(target=run_server, daemon=True)
    SERVER_THREAD.start()
    time.sleep(2)
    print("Uvicorn started on http://127.0.0.1:8000")
else:
    print("Server already running.")

print("Notebook setup complete. Use the printed trycloudflare URL to access web UI.")